In [1]:
#!/usr/bin/env python3

"""
Wildfire-centered temporal disturbance analysis.

For every wildfire pixel-year from 2000-2020, characterize the occurrence
of biotic disturbance (BT) and hotter drought (HD) from 10 years before
through 10 years after wildfire.

Analyses are performed separately for:
    1. Any disturbance thresholds
    2. Extreme disturbance thresholds

Important:
- Every wildfire pixel-year is treated as an event.
- A pixel that burns more than once contributes multiple wildfire events.
- Years outside 2000-2020 are treated as unobservable, NOT as zero disturbance.
- All calculations use the native 30-m data.
- Outputs are summaries for later visualization in R.
"""

from pathlib import Path
from collections import Counter, defaultdict
import csv

import numpy as np
import rasterio


################################################################################
# PATHS
################################################################################

PROJECT_DIR = Path(
    "/home/jovyan/data-store/forest-disturbance-stack-v3"
)

BINARY_DIR = (
    PROJECT_DIR /
    "data/derived/annual_stacks/binary"
)

FOREST_MASK = (
    PROJECT_DIR /
    "data/derived/resampled/forest_mask_30m_resampled.tif"
)

ECO_RASTER = (
    PROJECT_DIR /
    "data/derived/ecoregions/ecoregions_level3_30m.tif"
)

OUT_DIR = (
    PROJECT_DIR /
    "data/derived/disturbance_sequence/wildfire_context"
)

SUMMARY_DIR = (
    PROJECT_DIR /
    "data/derived/disturbance_sequence/summaries"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


################################################################################
# SETTINGS
################################################################################

YEARS = list(
    range(2000, 2021)
)

THRESHOLDS = [
    "any",
    "extreme"
]

# Annual binary stack:
# band 1 = wildfire
# band 2 = biotic disturbance
# band 3 = hotter drought
# band 4 = PDSI

WF_BAND = 1
BT_BAND = 2
HD_BAND = 3

MAX_LAG = 10

WINDOWS = [
    1,
    3,
    5
]

PIXEL_AREA_KM2 = 0.0009

NODATA_UINT8 = 255


################################################################################
# INPUT PATHS
################################################################################

def annual_stack_path(threshold, year):

    return (
        BINARY_DIR /
        f"annual_stack_{threshold}_{year}.tif"
    )


################################################################################
# VALIDATE INPUTS
################################################################################

print("Validating inputs...")

if not FOREST_MASK.exists():
    raise FileNotFoundError(FOREST_MASK)

if not ECO_RASTER.exists():
    raise FileNotFoundError(ECO_RASTER)

for threshold in THRESHOLDS:

    for year in YEARS:

        path = annual_stack_path(
            threshold,
            year
        )

        if not path.exists():
            raise FileNotFoundError(path)


################################################################################
# REFERENCE GRID
################################################################################

reference_path = annual_stack_path(
    "any",
    2000
)

with rasterio.open(reference_path) as ref:

    ref_width = ref.width
    ref_height = ref.height
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_profile = ref.profile.copy()


def check_alignment(path):

    with rasterio.open(path) as src:

        if (
            src.width != ref_width or
            src.height != ref_height or
            src.transform != ref_transform or
            src.crs != ref_crs
        ):

            raise ValueError(
                f"Raster alignment mismatch: {path}"
            )


check_alignment(FOREST_MASK)
check_alignment(ECO_RASTER)

for threshold in THRESHOLDS:
    for year in YEARS:
        check_alignment(
            annual_stack_path(
                threshold,
                year
            )
        )

print("All inputs are aligned.")


################################################################################
# OUTPUT RASTER PROFILE
################################################################################

frequency_profile = ref_profile.copy()

frequency_profile.update(
    count=1,
    dtype="uint8",
    nodata=NODATA_UINT8,
    compress="DEFLATE",
    predictor=2,
    tiled=True,
    BIGTIFF="YES"
)


################################################################################
# GLOBAL SUMMARY COUNTERS
################################################################################

# Relative-year analysis
#
# Key:
# threshold, context disturbance, relative year
#
# Values:
# number of wildfire pixel-events eligible for that relative year
# number experiencing context disturbance

relative_eligible = Counter()
relative_disturbed = Counter()

# Same thing by ecoregion

relative_eligible_eco = Counter()
relative_disturbed_eco = Counter()


# Window summaries
#
# We will accumulate distributions rather than just means.
#
# Key:
# threshold, context, direction, window, n_context_years

window_count_distribution = Counter()
window_count_distribution_eco = Counter()


# Presence within window
#
# Key:
# threshold, context, direction, window

window_eligible = Counter()
window_present = Counter()

window_eligible_eco = Counter()
window_present_eco = Counter()


# Same-year disturbance

same_year_eligible = Counter()
same_year_present = Counter()

same_year_eligible_eco = Counter()
same_year_present_eco = Counter()


# Nearest occurrence
#
# Key:
# threshold, context, direction, lag

nearest_lag_counts = Counter()
nearest_lag_counts_eco = Counter()


# Wildfire recurrence

wildfire_frequency_counts = {}
wildfire_frequency_counts_eco = {}


################################################################################
# PROCESS ANY AND EXTREME
################################################################################

for threshold in THRESHOLDS:

    print()
    print("=" * 80)
    print(
        f"PROCESSING {threshold.upper()} WILDFIRE CONTEXT"
    )
    print("=" * 80)


    ########################################################################
    # OPEN ANNUAL RASTERS
    ########################################################################

    annual_sources = {
        year: rasterio.open(
            annual_stack_path(
                threshold,
                year
            )
        )
        for year in YEARS
    }

    forest_src = rasterio.open(
        FOREST_MASK
    )

    eco_src = rasterio.open(
        ECO_RASTER
    )


    ########################################################################
    # WILDFIRE FREQUENCY OUTPUT
    ########################################################################

    threshold_out_dir = (
        OUT_DIR /
        threshold
    )

    threshold_out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    wildfire_frequency_path = (
        threshold_out_dir /
        "wildfire_frequency.tif"
    )

    wildfire_frequency_dst = rasterio.open(
        wildfire_frequency_path,
        "w",
        **frequency_profile
    )


    ########################################################################
    # RECURRENCE COUNTERS
    ########################################################################

    wf_frequency_counter = Counter()
    wf_frequency_eco_counter = Counter()


    ########################################################################
    # PROCESS BLOCKS
    ########################################################################

    block_number = 0

    for block_index, window in forest_src.block_windows(1):

        block_number += 1

        if block_number % 100 == 0:
            print(
                f"  Processed {block_number} blocks..."
            )


        ####################################################################
        # FOREST + ECOREGION
        ####################################################################

        forest = (
            forest_src.read(
                1,
                window=window
            ) == 1
        )

        eco = eco_src.read(
            1,
            window=window
        )


        ####################################################################
        # READ THE COMPLETE 21-YEAR TIME SERIES FOR THIS BLOCK
        ####################################################################

        wf_by_year = {}
        bt_by_year = {}
        hd_by_year = {}


        for year in YEARS:

            src = annual_sources[
                year
            ]

            wf_by_year[year] = (
                (
                    src.read(
                        WF_BAND,
                        window=window
                    ) > 0
                ) &
                forest
            )

            bt_by_year[year] = (
                (
                    src.read(
                        BT_BAND,
                        window=window
                    ) > 0
                ) &
                forest
            )

            hd_by_year[year] = (
                (
                    src.read(
                        HD_BAND,
                        window=window
                    ) > 0
                ) &
                forest
            )


        ####################################################################
        # WILDFIRE FREQUENCY
        ####################################################################

        wildfire_frequency = np.zeros(
            forest.shape,
            dtype=np.uint8
        )

        for year in YEARS:

            wildfire_frequency += (
                wf_by_year[year]
                .astype(np.uint8)
            )


        wildfire_frequency_out = (
            wildfire_frequency.copy()
        )

        wildfire_frequency_out[
            ~forest
        ] = NODATA_UINT8


        wildfire_frequency_dst.write(
            wildfire_frequency_out,
            1,
            window=window
        )


        ####################################################################
        # WESTWIDE WILDFIRE RECURRENCE
        ####################################################################

        frequencies, counts = np.unique(
            wildfire_frequency[
                forest
            ],
            return_counts=True
        )

        for frequency, count in zip(
            frequencies,
            counts
        ):

            wf_frequency_counter[
                int(frequency)
            ] += int(count)


        ####################################################################
        # ECOREGION WILDFIRE RECURRENCE
        ####################################################################

        eco_ids = np.unique(
            eco[forest]
        )

        eco_ids = [
            int(x)
            for x in eco_ids
            if x > 0
        ]


        for eco_id in eco_ids:

            eco_mask = (
                forest &
                (eco == eco_id)
            )

            frequencies, counts = np.unique(
                wildfire_frequency[
                    eco_mask
                ],
                return_counts=True
            )

            for frequency, count in zip(
                frequencies,
                counts
            ):

                wf_frequency_eco_counter[
                    (
                        eco_id,
                        int(frequency)
                    )
                ] += int(count)


        ####################################################################
        # WILDFIRE-CENTERED ANALYSIS
        ####################################################################

        for wf_year in YEARS:

            wf_event = (
                wf_by_year[
                    wf_year
                ]
            )

            n_wf_events = int(
                np.count_nonzero(
                    wf_event
                )
            )

            if n_wf_events == 0:
                continue


            ################################################################
            # ECOREGIONS CONTAINING THIS WILDFIRE EVENT
            ################################################################

            event_eco_ids = np.unique(
                eco[
                    wf_event
                ]
            )

            event_eco_ids = [
                int(x)
                for x in event_eco_ids
                if x > 0
            ]


            ################################################################
            # SAME-YEAR BT AND HD
            ################################################################

            for context_name, context_data in [
                ("BT", bt_by_year),
                ("HD", hd_by_year)
            ]:

                context_same_year = (
                    context_data[
                        wf_year
                    ]
                )


                same_year_eligible[
                    (
                        threshold,
                        context_name
                    )
                ] += n_wf_events


                same_year_present[
                    (
                        threshold,
                        context_name
                    )
                ] += int(
                    np.count_nonzero(
                        wf_event &
                        context_same_year
                    )
                )


                ############################################################
                # SAME-YEAR BY ECOREGION
                ############################################################

                for eco_id in event_eco_ids:

                    eco_event = (
                        wf_event &
                        (eco == eco_id)
                    )

                    n_eco_events = int(
                        np.count_nonzero(
                            eco_event
                        )
                    )

                    same_year_eligible_eco[
                        (
                            threshold,
                            eco_id,
                            context_name
                        )
                    ] += n_eco_events


                    same_year_present_eco[
                        (
                            threshold,
                            eco_id,
                            context_name
                        )
                    ] += int(
                        np.count_nonzero(
                            eco_event &
                            context_same_year
                        )
                    )


            ################################################################
            # RELATIVE YEAR: -10 THROUGH +10
            ################################################################

            for relative_year in range(
                -MAX_LAG,
                MAX_LAG + 1
            ):

                target_year = (
                    wf_year +
                    relative_year
                )

                # Outside observation period:
                # unobservable, not zero.

                if target_year not in YEARS:
                    continue


                for context_name, context_data in [
                    ("BT", bt_by_year),
                    ("HD", hd_by_year)
                ]:

                    context_event = (
                        context_data[
                            target_year
                        ]
                    )


                    relative_eligible[
                        (
                            threshold,
                            context_name,
                            relative_year
                        )
                    ] += n_wf_events


                    relative_disturbed[
                        (
                            threshold,
                            context_name,
                            relative_year
                        )
                    ] += int(
                        np.count_nonzero(
                            wf_event &
                            context_event
                        )
                    )


                    ########################################################
                    # ECOREGION RELATIVE-YEAR SUMMARY
                    ########################################################

                    for eco_id in event_eco_ids:

                        eco_event = (
                            wf_event &
                            (eco == eco_id)
                        )

                        n_eco_events = int(
                            np.count_nonzero(
                                eco_event
                            )
                        )


                        relative_eligible_eco[
                            (
                                threshold,
                                eco_id,
                                context_name,
                                relative_year
                            )
                        ] += n_eco_events


                        relative_disturbed_eco[
                            (
                                threshold,
                                eco_id,
                                context_name,
                                relative_year
                            )
                        ] += int(
                            np.count_nonzero(
                                eco_event &
                                context_event
                            )
                        )


            ################################################################
            # PRE/POST WINDOWS
            ################################################################

            for context_name, context_data in [
                ("BT", bt_by_year),
                ("HD", hd_by_year)
            ]:

                for window_size in WINDOWS:

                    for direction in [
                        "pre",
                        "post"
                    ]:


                        ####################################################
                        # DEFINE YEARS IN WINDOW
                        ####################################################

                        if direction == "pre":

                            requested_years = list(
                                range(
                                    wf_year - window_size,
                                    wf_year
                                )
                            )

                        else:

                            requested_years = list(
                                range(
                                    wf_year + 1,
                                    wf_year + window_size + 1
                                )
                            )


                        # Require the FULL requested window.
                        #
                        # Example:
                        # a 2002 wildfire does not contribute to the
                        # 5-year pre-fire analysis.

                        if not all(
                            year in YEARS
                            for year in requested_years
                        ):
                            continue


                        ####################################################
                        # NUMBER OF CONTEXT-DISTURBANCE YEARS
                        ####################################################

                        context_count = np.zeros(
                            forest.shape,
                            dtype=np.uint8
                        )


                        for year in requested_years:

                            context_count += (
                                context_data[
                                    year
                                ].astype(np.uint8)
                            )


                        event_counts = (
                            context_count[
                                wf_event
                            ]
                        )


                        ####################################################
                        # ELIGIBLE EVENTS
                        ####################################################

                        window_eligible[
                            (
                                threshold,
                                context_name,
                                direction,
                                window_size
                            )
                        ] += n_wf_events


                        ####################################################
                        # AT LEAST ONE CONTEXT EVENT
                        ####################################################

                        window_present[
                            (
                                threshold,
                                context_name,
                                direction,
                                window_size
                            )
                        ] += int(
                            np.count_nonzero(
                                event_counts > 0
                            )
                        )


                        ####################################################
                        # FULL COUNT DISTRIBUTION
                        ####################################################

                        values, counts = np.unique(
                            event_counts,
                            return_counts=True
                        )

                        for value, count in zip(
                            values,
                            counts
                        ):

                            window_count_distribution[
                                (
                                    threshold,
                                    context_name,
                                    direction,
                                    window_size,
                                    int(value)
                                )
                            ] += int(count)


                        ####################################################
                        # ECOREGION WINDOW SUMMARIES
                        ####################################################

                        for eco_id in event_eco_ids:

                            eco_event = (
                                wf_event &
                                (eco == eco_id)
                            )

                            n_eco_events = int(
                                np.count_nonzero(
                                    eco_event
                                )
                            )

                            eco_counts = (
                                context_count[
                                    eco_event
                                ]
                            )


                            window_eligible_eco[
                                (
                                    threshold,
                                    eco_id,
                                    context_name,
                                    direction,
                                    window_size
                                )
                            ] += n_eco_events


                            window_present_eco[
                                (
                                    threshold,
                                    eco_id,
                                    context_name,
                                    direction,
                                    window_size
                                )
                            ] += int(
                                np.count_nonzero(
                                    eco_counts > 0
                                )
                            )


                            values, counts = np.unique(
                                eco_counts,
                                return_counts=True
                            )

                            for value, count in zip(
                                values,
                                counts
                            ):

                                window_count_distribution_eco[
                                    (
                                        threshold,
                                        eco_id,
                                        context_name,
                                        direction,
                                        window_size,
                                        int(value)
                                    )
                                ] += int(count)


            ################################################################
            # NEAREST BT / HD BEFORE AND AFTER FIRE
            ################################################################

            for context_name, context_data in [
                ("BT", bt_by_year),
                ("HD", hd_by_year)
            ]:

                for direction in [
                    "pre",
                    "post"
                ]:

                    nearest = np.zeros(
                        forest.shape,
                        dtype=np.uint8
                    )


                    # Search from nearest to farthest.

                    for lag in range(
                        1,
                        MAX_LAG + 1
                    ):

                        if direction == "pre":
                            target_year = wf_year - lag

                        else:
                            target_year = wf_year + lag


                        if target_year not in YEARS:
                            continue


                        match = (
                            wf_event &
                            (nearest == 0) &
                            context_data[
                                target_year
                            ]
                        )

                        nearest[
                            match
                        ] = lag


                    ########################################################
                    # RECORD ONLY EVENTS WITH AN OBSERVED CONTEXT EVENT
                    ########################################################

                    observed = (
                        wf_event &
                        (nearest > 0)
                    )

                    values, counts = np.unique(
                        nearest[
                            observed
                        ],
                        return_counts=True
                    )


                    for lag, count in zip(
                        values,
                        counts
                    ):

                        nearest_lag_counts[
                            (
                                threshold,
                                context_name,
                                direction,
                                int(lag)
                            )
                        ] += int(count)


                    ########################################################
                    # ECOREGION NEAREST LAGS
                    ########################################################

                    for eco_id in event_eco_ids:

                        eco_observed = (
                            observed &
                            (eco == eco_id)
                        )

                        values, counts = np.unique(
                            nearest[
                                eco_observed
                            ],
                            return_counts=True
                        )


                        for lag, count in zip(
                            values,
                            counts
                        ):

                            nearest_lag_counts_eco[
                                (
                                    threshold,
                                    eco_id,
                                    context_name,
                                    direction,
                                    int(lag)
                                )
                            ] += int(count)


    ########################################################################
    # CLOSE FILES
    ########################################################################

    wildfire_frequency_dst.close()

    forest_src.close()
    eco_src.close()

    for src in annual_sources.values():
        src.close()


    wildfire_frequency_counts[
        threshold
    ] = wf_frequency_counter

    wildfire_frequency_counts_eco[
        threshold
    ] = wf_frequency_eco_counter


    print(
        f"Finished {threshold.upper()}."
    )


################################################################################
# WRITE RELATIVE-YEAR WESTWIDE CSV
################################################################################

relative_file = (
    SUMMARY_DIR /
    "wildfire_context_relative_year.csv"
)

with open(
    relative_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "relative_year",
        "wildfire_pixel_events_eligible",
        "context_pixel_events",
        "percent"
    ])


    for key in sorted(relative_eligible):

        threshold, context, relative_year = key

        eligible = relative_eligible[key]

        disturbed = relative_disturbed[key]

        percent = (
            disturbed /
            eligible *
            100
            if eligible > 0
            else np.nan
        )


        writer.writerow([
            threshold,
            context,
            relative_year,
            eligible,
            disturbed,
            percent
        ])


################################################################################
# WRITE RELATIVE-YEAR ECOREGION CSV
################################################################################

relative_eco_file = (
    SUMMARY_DIR /
    "wildfire_context_relative_year_ecoregions.csv"
)

with open(
    relative_eco_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "ecoregion_id",
        "context_disturbance",
        "relative_year",
        "wildfire_pixel_events_eligible",
        "context_pixel_events",
        "percent"
    ])


    for key in sorted(
        relative_eligible_eco
    ):

        (
            threshold,
            eco_id,
            context,
            relative_year
        ) = key

        eligible = (
            relative_eligible_eco[
                key
            ]
        )

        disturbed = (
            relative_disturbed_eco[
                key
            ]
        )

        percent = (
            disturbed /
            eligible *
            100
            if eligible > 0
            else np.nan
        )


        writer.writerow([
            threshold,
            eco_id,
            context,
            relative_year,
            eligible,
            disturbed,
            percent
        ])


################################################################################
# WRITE WINDOW SUMMARY
################################################################################

window_file = (
    SUMMARY_DIR /
    "wildfire_context_windows.csv"
)

with open(
    window_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "direction",
        "window_years",
        "wildfire_pixel_events_eligible",
        "pixel_events_with_context",
        "percent_with_context"
    ])


    for key in sorted(
        window_eligible
    ):

        (
            threshold,
            context,
            direction,
            window_size
        ) = key

        eligible = (
            window_eligible[key]
        )

        present = (
            window_present[key]
        )


        writer.writerow([
            threshold,
            context,
            direction,
            window_size,
            eligible,
            present,
            (
                present /
                eligible *
                100
                if eligible > 0
                else np.nan
            )
        ])


################################################################################
# WRITE WINDOW COUNT DISTRIBUTIONS
################################################################################

window_distribution_file = (
    SUMMARY_DIR /
    "wildfire_context_window_distributions.csv"
)

with open(
    window_distribution_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "direction",
        "window_years",
        "n_context_years",
        "wildfire_pixel_events"
    ])


    for key, count in sorted(
        window_count_distribution.items()
    ):

        (
            threshold,
            context,
            direction,
            window_size,
            n_context_years
        ) = key


        writer.writerow([
            threshold,
            context,
            direction,
            window_size,
            n_context_years,
            count
        ])


################################################################################
# WRITE ECOREGION WINDOW SUMMARY
################################################################################

window_eco_file = (
    SUMMARY_DIR /
    "wildfire_context_windows_ecoregions.csv"
)

with open(
    window_eco_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "ecoregion_id",
        "context_disturbance",
        "direction",
        "window_years",
        "wildfire_pixel_events_eligible",
        "pixel_events_with_context",
        "percent_with_context"
    ])


    for key in sorted(
        window_eligible_eco
    ):

        (
            threshold,
            eco_id,
            context,
            direction,
            window_size
        ) = key

        eligible = (
            window_eligible_eco[key]
        )

        present = (
            window_present_eco[key]
        )


        writer.writerow([
            threshold,
            eco_id,
            context,
            direction,
            window_size,
            eligible,
            present,
            (
                present /
                eligible *
                100
                if eligible > 0
                else np.nan
            )
        ])


################################################################################
# WRITE ECOREGION WINDOW COUNT DISTRIBUTIONS
################################################################################

window_distribution_eco_file = (
    SUMMARY_DIR /
    "wildfire_context_window_distributions_ecoregions.csv"
)

with open(
    window_distribution_eco_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "ecoregion_id",
        "context_disturbance",
        "direction",
        "window_years",
        "n_context_years",
        "wildfire_pixel_events"
    ])


    for key, count in sorted(
        window_count_distribution_eco.items()
    ):

        (
            threshold,
            eco_id,
            context,
            direction,
            window_size,
            n_context_years
        ) = key


        writer.writerow([
            threshold,
            eco_id,
            context,
            direction,
            window_size,
            n_context_years,
            count
        ])


################################################################################
# WRITE SAME-YEAR SUMMARY
################################################################################

same_year_file = (
    SUMMARY_DIR /
    "wildfire_context_same_year.csv"
)

with open(
    same_year_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "wildfire_pixel_events",
        "same_year_pixel_events",
        "percent_same_year"
    ])


    for key in sorted(
        same_year_eligible
    ):

        threshold, context = key

        eligible = (
            same_year_eligible[key]
        )

        present = (
            same_year_present[key]
        )


        writer.writerow([
            threshold,
            context,
            eligible,
            present,
            (
                present /
                eligible *
                100
                if eligible > 0
                else np.nan
            )
        ])


################################################################################
# WRITE NEAREST-LAG SUMMARY
################################################################################

nearest_file = (
    SUMMARY_DIR /
    "wildfire_context_nearest_lags.csv"
)

with open(
    nearest_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "direction",
        "lag_years",
        "wildfire_pixel_events"
    ])


    for key, count in sorted(
        nearest_lag_counts.items()
    ):

        (
            threshold,
            context,
            direction,
            lag
        ) = key


        writer.writerow([
            threshold,
            context,
            direction,
            lag,
            count
        ])


################################################################################
# WRITE WILDFIRE RECURRENCE SUMMARY
################################################################################

reburn_file = (
    SUMMARY_DIR /
    "wildfire_reburn_summary.csv"
)

with open(
    reburn_file,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "wildfire_year_count",
        "pixel_count",
        "area_km2"
    ])


    for threshold in THRESHOLDS:

        counter = (
            wildfire_frequency_counts[
                threshold
            ]
        )

        for frequency in sorted(
            counter
        ):

            count = counter[
                frequency
            ]

            writer.writerow([
                threshold,
                frequency,
                count,
                count * PIXEL_AREA_KM2
            ])


################################################################################
# FINISHED
################################################################################

print()
print("=" * 80)
print("WILDFIRE-CENTERED ANALYSIS COMPLETE")
print("=" * 80)

print()
print("Main outputs:")

for path in [
    relative_file,
    relative_eco_file,
    window_file,
    window_distribution_file,
    window_eco_file,
    window_distribution_eco_file,
    same_year_file,
    nearest_file,
    reburn_file
]:

    print(
        f"  {path}"
    )

Validating inputs...
All inputs are aligned.

PROCESSING ANY WILDFIRE CONTEXT
  Processed 100 blocks...
  Processed 200 blocks...
  Processed 300 blocks...
  Processed 400 blocks...
  Processed 500 blocks...
  Processed 600 blocks...
  Processed 700 blocks...
  Processed 800 blocks...
  Processed 900 blocks...
  Processed 1000 blocks...
  Processed 1100 blocks...
  Processed 1200 blocks...
  Processed 1300 blocks...
  Processed 1400 blocks...
  Processed 1500 blocks...
  Processed 1600 blocks...
  Processed 1700 blocks...
  Processed 1800 blocks...
  Processed 1900 blocks...
  Processed 2000 blocks...
  Processed 2100 blocks...
  Processed 2200 blocks...
  Processed 2300 blocks...
  Processed 2400 blocks...
  Processed 2500 blocks...
  Processed 2600 blocks...
  Processed 2700 blocks...
  Processed 2800 blocks...
  Processed 2900 blocks...
  Processed 3000 blocks...
  Processed 3100 blocks...
  Processed 3200 blocks...
  Processed 3300 blocks...
  Processed 3400 blocks...
  Processed 3

In [3]:
################################################################################
# BALANCED WILDFIRE-CENTERED ANALYSIS: ±5 YEARS
#
# Focal wildfire years: 2005–2015
# Context disturbances: BT and HD
# Thresholds: Any and Extreme
#
# Produces:
#   wildfire_context_balanced_5yr.csv
#   wildfire_context_balanced_5yr_ecoregions.csv
#
# This cell runs independently of the previous analysis.
################################################################################

from pathlib import Path
from collections import Counter
from contextlib import ExitStack
import csv

import numpy as np
import rasterio


# ==============================================================================
# 1. PATHS AND SETTINGS
# ==============================================================================

PROJECT_DIR = Path(
    "/home/jovyan/data-store/forest-disturbance-stack-v3"
)

BINARY_DIR = PROJECT_DIR / "data/derived/annual_stacks/binary"

FOREST_MASK = (
    PROJECT_DIR /
    "data/derived/resampled/forest_mask_30m_resampled.tif"
)

ECO_RASTER = (
    PROJECT_DIR /
    "data/derived/ecoregions/ecoregions_level3_30m.tif"
)

SUMMARY_DIR = (
    PROJECT_DIR /
    "data/derived/disturbance_sequence/summaries"
)

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS = ["any", "extreme"]

BALANCED_LAG = 5

FOCAL_YEARS = list(range(2005, 2016))

CONTEXT_BANDS = {
    "BT": 2,
    "HD": 3
}


def annual_path(threshold, year):
    return BINARY_DIR / f"annual_stack_{threshold}_{year}.tif"


# ==============================================================================
# 2. COUNTERS
# ==============================================================================

# Westwide:
# (threshold, context, relative_year)

eligible_west = Counter()
disturbed_west = Counter()

# Ecoregions:
# (threshold, ecoregion_id, context, relative_year)

eligible_eco = Counter()
disturbed_eco = Counter()


# ==============================================================================
# 3. PROCESS EACH THRESHOLD
# ==============================================================================

for threshold in THRESHOLDS:

    print(f"\nProcessing balanced analysis: {threshold.upper()}")

    # We need all years from 2000–2020 because the first/last
    # focal years have five-year windows extending to those dates.

    with ExitStack() as stack:

        annual = {
            year: stack.enter_context(
                rasterio.open(annual_path(threshold, year))
            )
            for year in range(2000, 2021)
        }

        forest_src = stack.enter_context(
            rasterio.open(FOREST_MASK)
        )

        eco_src = stack.enter_context(
            rasterio.open(ECO_RASTER)
        )

        # Validate grids before processing.

        reference = annual[2000]

        for src in [forest_src, eco_src, *annual.values()]:

            if (
                src.width != reference.width
                or src.height != reference.height
                or src.transform != reference.transform
                or src.crs != reference.crs
            ):
                raise ValueError(
                    f"Raster grid mismatch: {src.name}"
                )

        # Use native raster blocks.

        for block_number, (block_index, window) in enumerate(
            forest_src.block_windows(1), start=1
):

            if block_number % 100 == 0:
                print(f"  Processed {block_number:,} blocks...")

            forest = (
                forest_src.read(1, window=window) == 1
            )

            if not np.any(forest):
                continue

            eco = eco_src.read(1, window=window)

            # Read annual BT and HD once per block.

            context = {}

            for year in range(2000, 2021):

                bands = annual[year].read(
                    [2, 3],
                    window=window
                )

                context[year] = {
                    "BT": (bands[0] > 0) & forest,
                    "HD": (bands[1] > 0) & forest
                }

            # Each focal wildfire year represents a distinct
            # wildfire pixel-event.

            for wf_year in FOCAL_YEARS:

                wf = (
                    (annual[wf_year].read(1, window=window) > 0)
                    & forest
                )

                n_events = int(np.count_nonzero(wf))

                if n_events == 0:
                    continue

                # Ecoregion event counts, calculated once.

                eco_values, eco_counts = np.unique(
                    eco[wf],
                    return_counts=True
                )

                eco_event_counts = {
                    int(e): int(n)
                    for e, n in zip(eco_values, eco_counts)
                    if e > 0 and (
                        eco_src.nodata is None or e != eco_src.nodata
                    )
                }

                # Calculate relative-year disturbance prevalence.

                for relative_year in range(
                    -BALANCED_LAG,
                    BALANCED_LAG + 1
                ):

                    target_year = wf_year + relative_year

                    for context_name in CONTEXT_BANDS:

                        context_mask = context[target_year][context_name]

                        key = (
                            threshold,
                            context_name,
                            relative_year
                        )

                        eligible_west[key] += n_events

                        overlap = wf & context_mask

                        disturbed_west[key] += int(
                            np.count_nonzero(overlap)
                        )

                        # Ecoregion summaries

                        overlap_eco, overlap_counts = np.unique(
                            eco[overlap],
                            return_counts=True
                        )

                        overlap_lookup = {
                            int(e): int(n)
                            for e, n in zip(
                                overlap_eco,
                                overlap_counts
                            )
                        }

                        for eco_id, eco_n in eco_event_counts.items():

                            eco_key = (
                                threshold,
                                eco_id,
                                context_name,
                                relative_year
                            )

                            eligible_eco[eco_key] += eco_n

                            disturbed_eco[eco_key] += (
                                overlap_lookup.get(eco_id, 0)
                            )

    print(f"Finished {threshold.upper()}")


# ==============================================================================
# 4. WRITE WESTWIDE CSV
# ==============================================================================

west_file = (
    SUMMARY_DIR /
    "wildfire_context_balanced_5yr.csv"
)

with open(west_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "context_disturbance",
        "relative_year",
        "wildfire_pixel_events_eligible",
        "context_pixel_events",
        "percent"
    ])

    for key in sorted(eligible_west):

        threshold, context_name, relative_year = key

        eligible = eligible_west[key]
        disturbed = disturbed_west[key]

        writer.writerow([
            threshold,
            context_name,
            relative_year,
            eligible,
            disturbed,
            100 * disturbed / eligible if eligible else np.nan
        ])


# ==============================================================================
# 5. WRITE ECOREGION CSV
# ==============================================================================

eco_file = (
    SUMMARY_DIR /
    "wildfire_context_balanced_5yr_ecoregions.csv"
)

with open(eco_file, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow([
        "threshold",
        "ecoregion_id",
        "context_disturbance",
        "relative_year",
        "wildfire_pixel_events_eligible",
        "context_pixel_events",
        "percent"
    ])

    for key in sorted(eligible_eco):

        threshold, eco_id, context_name, relative_year = key

        eligible = eligible_eco[key]
        disturbed = disturbed_eco[key]

        writer.writerow([
            threshold,
            eco_id,
            context_name,
            relative_year,
            eligible,
            disturbed,
            100 * disturbed / eligible if eligible else np.nan
        ])


# ==============================================================================
# 6. VALIDATE BALANCED DENOMINATORS
# ==============================================================================

print("\nValidating balanced denominators...")

for threshold in THRESHOLDS:

    for context_name in CONTEXT_BANDS:

        denominators = {
            eligible_west[
                (threshold, context_name, relative_year)
            ]
            for relative_year in range(
                -BALANCED_LAG,
                BALANCED_LAG + 1
            )
        }

        if len(denominators) != 1:
            raise AssertionError(
                f"Denominators vary for {threshold}, {context_name}"
            )

        print(
            f"  {threshold.upper()} / {context_name}: "
            f"{next(iter(denominators)):,} wildfire pixel-events"
        )


print("\nBalanced analysis complete.")

print(f"\nWestwide CSV: {west_file}")
print(f"Ecoregion CSV: {eco_file}")


Processing balanced analysis: ANY
  Processed 100 blocks...
  Processed 200 blocks...
  Processed 300 blocks...
  Processed 400 blocks...
  Processed 500 blocks...
  Processed 600 blocks...
  Processed 700 blocks...
  Processed 800 blocks...
  Processed 900 blocks...
  Processed 1,000 blocks...
  Processed 1,100 blocks...
  Processed 1,200 blocks...
  Processed 1,300 blocks...
  Processed 1,400 blocks...
  Processed 1,500 blocks...
  Processed 1,600 blocks...
  Processed 1,700 blocks...
  Processed 1,800 blocks...
  Processed 1,900 blocks...
  Processed 2,000 blocks...
  Processed 2,100 blocks...
  Processed 2,200 blocks...
  Processed 2,300 blocks...
  Processed 2,400 blocks...
  Processed 2,500 blocks...
  Processed 2,600 blocks...
  Processed 2,700 blocks...
  Processed 2,800 blocks...
  Processed 2,900 blocks...
  Processed 3,000 blocks...
  Processed 3,100 blocks...
  Processed 3,200 blocks...
  Processed 3,300 blocks...
  Processed 3,400 blocks...
  Processed 3,500 blocks...
  P

In [4]:
################################################################################
# Forest Disturbance Stack
# Extreme disturbance sequences and transitions, 2000–2020
#
# Outputs:
#   - First-observed-year rasters for WF, BT, HD
#   - 25-class sequence raster
#   - Sequence codebook
#   - Westwide and ecoregion sequence summaries
#   - Disturbance-count summary (0–3 types)
#   - Consecutive transition summaries
#   - Transition-lag distributions
#   - Alluvial-ready CSV
################################################################################

from pathlib import Path
from contextlib import ExitStack
from itertools import combinations, product
from collections import Counter
import csv

import numpy as np
import rasterio


# =============================================================================
# 1. DIRECTORIES AND SETTINGS
# =============================================================================

ROOT = Path(
    "/home/jovyan/data-store/forest-disturbance-stack-v3"
)

ANNUAL_DIR = ROOT / "data/derived/annual_stacks/binary"

FOREST_PATH = (
    ROOT / "data/derived/resampled/"
    "forest_mask_30m_resampled.tif"
)

ECO_PATH = (
    ROOT / "data/derived/ecoregions/"
    "ecoregions_level3_30m.tif"
)

OUT_DIR = (
    ROOT / "data/derived/disturbance_sequence/"
    "extreme_sequences"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2000, 2021)

DISTURBANCES = ("WF", "BT", "HD")

# Band order in annual binary stacks (1-based)
BANDS = {
    "WF": 1,
    "BT": 2,
    "HD": 3
}

PIXEL_AREA_KM2 = 0.0009

MISSING_YEAR = 32767
OUTPUT_YEAR_NODATA = -9999

print("Output directory:", OUT_DIR)


# =============================================================================
# 2. GENERATE ALL 25 FIRST-OBSERVED SEQUENCES
# =============================================================================

def ordered_partitions(items):
    """
    Generate all unique ordered partitions of a set.

    Each partition represents a first-observed sequence.
    Multiple disturbances within a group occurred in the same year.
    """

    n = len(items)
    seen = set()

    for ranks in product(range(n), repeat=n):

        unique_ranks = sorted(set(ranks))

        normalized = tuple(
            unique_ranks.index(r)
            for r in ranks
        )

        if normalized in seen:
            continue

        seen.add(normalized)

        groups = tuple(
            tuple(
                items[i]
                for i in range(n)
                if normalized[i] == rank
            )
            for rank in range(len(unique_ranks))
        )

        yield groups


def stage_label(group):
    return " + ".join(group)


def sequence_label(groups):
    return " → ".join(
        stage_label(group)
        for group in groups
    )


# Sequence code 0 = no extreme disturbance within forest.
# Codes 1–25 = all possible nonempty sequences.

sequences = []

for n_types in (1, 2, 3):

    for subset in combinations(DISTURBANCES, n_types):

        sequences.extend(
            ordered_partitions(subset)
        )

assert len(sequences) == 25

CODE_TO_GROUPS = {
    code: groups
    for code, groups in enumerate(sequences, start=1)
}

CODE_TO_LABEL = {
    code: sequence_label(groups)
    for code, groups in CODE_TO_GROUPS.items()
}

CODE_TO_LABEL[0] = "No extreme disturbance"

CODE_TO_N_TYPES = {
    code: sum(len(group) for group in groups)
    for code, groups in CODE_TO_GROUPS.items()
}

CODE_TO_N_TYPES[0] = 0


def save_csv(filename, header, rows):

    path = OUT_DIR / filename

    with open(path, "w", newline="") as f:

        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)

    print("Saved:", path.name)


# Codebook
codebook_rows = []

for code in range(26):

    groups = CODE_TO_GROUPS.get(code, ())

    codebook_rows.append([
        code,
        CODE_TO_LABEL[code],
        CODE_TO_N_TYPES[code],
        len(groups),
        stage_label(groups[0]) if len(groups) >= 1 else "",
        stage_label(groups[1]) if len(groups) >= 2 else "",
        stage_label(groups[2]) if len(groups) >= 3 else ""
    ])

save_csv(
    "sequence_codebook.csv",
    [
        "sequence_code",
        "sequence",
        "n_disturbance_types",
        "n_stages",
        "stage_1",
        "stage_2",
        "stage_3"
    ],
    codebook_rows
)


# =============================================================================
# 3. CLASSIFY FIRST-OBSERVED SEQUENCES
# =============================================================================

def classify_sequences(first_year, forest):
    """
    Classify each forest pixel into one of 25 sequences.

    0 represents no extreme disturbance or outside forest.
    """

    classification = np.zeros(
        forest.shape,
        dtype=np.uint8
    )

    for code, groups in CODE_TO_GROUPS.items():

        present = {
            d
            for group in groups
            for d in group
        }

        mask = forest.copy()

        # Which disturbance types were observed?
        for d in DISTURBANCES:

            if d in present:
                mask &= (
                    first_year[d] != MISSING_YEAR
                )
            else:
                mask &= (
                    first_year[d] == MISSING_YEAR
                )

        # Same-year ties
        for group in groups:

            for d in group[1:]:

                mask &= (
                    first_year[d]
                    == first_year[group[0]]
                )

        # Chronological order between stages
        for earlier, later in zip(
            groups[:-1],
            groups[1:]
        ):

            mask &= (
                first_year[earlier[0]]
                < first_year[later[0]]
            )

        classification[mask] = code

    return classification


# =============================================================================
# 4. PREPARE SUMMARY COUNTERS
# =============================================================================

# Westwide pixel counts by sequence code
west_counts = Counter()

# Ecoregion counts:
# (ecoregion_id, sequence_code)
eco_counts = Counter()

# Ecoregion forest totals
eco_forest_counts = Counter()

# Consecutive transition counts:
# (sequence_code, transition_number, from_stage, to_stage)
transition_counts = Counter()

# Transition lag counts:
# (sequence_code, transition_number, from_stage, to_stage, lag)
transition_lag_counts = Counter()

forest_total = 0


# =============================================================================
# 5. PROCESS ANNUAL RASTERS
# =============================================================================

with ExitStack() as stack:

    forest_src = stack.enter_context(
        rasterio.open(FOREST_PATH)
    )

    eco_src = stack.enter_context(
        rasterio.open(ECO_PATH)
    )

    annual = {
        year: stack.enter_context(
            rasterio.open(
                ANNUAL_DIR /
                f"annual_stack_extreme_{year}.tif"
            )
        )
        for year in YEARS
    }

    # -------------------------------------------------------------------------
    # Validate grids
    # -------------------------------------------------------------------------

    for src in [eco_src, *annual.values()]:

        if not (
            src.width == forest_src.width
            and src.height == forest_src.height
            and src.transform == forest_src.transform
            and src.crs == forest_src.crs
        ):
            raise ValueError(
                f"Raster grid mismatch: {src.name}"
            )

    for year, src in annual.items():

        if src.count < 3:
            raise ValueError(
                f"Missing disturbance bands in {year}"
            )

    # -------------------------------------------------------------------------
    # Output raster profiles
    # -------------------------------------------------------------------------

    sequence_profile = forest_src.profile.copy()

    sequence_profile.update(
        driver="GTiff",
        count=1,
        dtype="uint8",
        nodata=0,
        compress="deflate",
        BIGTIFF="IF_SAFER"
    )

    year_profile = forest_src.profile.copy()

    year_profile.update(
        driver="GTiff",
        count=1,
        dtype="int16",
        nodata=OUTPUT_YEAR_NODATA,
        compress="deflate",
        BIGTIFF="IF_SAFER"
    )

    # Sequence classification raster
    seq_dst = stack.enter_context(
        rasterio.open(
            OUT_DIR / "extreme_sequence_class.tif",
            "w",
            **sequence_profile
        )
    )

    # First-observed-year rasters
    year_dsts = {
        d: stack.enter_context(
            rasterio.open(
                OUT_DIR / f"extreme_first_year_{d}.tif",
                "w",
                **year_profile
            )
        )
        for d in DISTURBANCES
    }

    # -------------------------------------------------------------------------
    # Block processing
    # -------------------------------------------------------------------------

    for block_number, (_, window) in enumerate(
        forest_src.block_windows(1),
        start=1
    ):

        if block_number % 100 == 0:

            print(
                f"Processed {block_number:,} blocks",
                flush=True
            )

        forest = (
            forest_src.read(1, window=window) == 1
        )

        shape = forest.shape

        # Skip blocks without forest
        if not forest.any():

            seq_dst.write(
                np.zeros(shape, dtype=np.uint8),
                1,
                window=window
            )

            for dst in year_dsts.values():

                dst.write(
                    np.full(
                        shape,
                        OUTPUT_YEAR_NODATA,
                        dtype=np.int16
                    ),
                    1,
                    window=window
                )

            continue

        eco = eco_src.read(
            1,
            window=window
        )

        valid_eco = (eco > 0)

        if eco_src.nodata is not None:

            valid_eco &= (
                eco != eco_src.nodata
            )

        # ---------------------------------------------------------------------
        # Find first observed year of each disturbance
        # ---------------------------------------------------------------------

        first_year = {
            d: np.full(
                shape,
                MISSING_YEAR,
                dtype=np.int16
            )
            for d in DISTURBANCES
        }

        for year in YEARS:

            bands = annual[year].read(
                [1, 2, 3],
                window=window
            )

            for i, d in enumerate(DISTURBANCES):

                new_pixels = (
                    (bands[i] == 1)
                    & forest
                    & (first_year[d] == MISSING_YEAR)
                )

                first_year[d][new_pixels] = year

        # ---------------------------------------------------------------------
        # Sequence classification
        # ---------------------------------------------------------------------

        classification = classify_sequences(
            first_year,
            forest
        )

        any_disturbance = np.zeros(
            shape,
            dtype=bool
        )

        for d in DISTURBANCES:

            any_disturbance |= (
                first_year[d] != MISSING_YEAR
            )

        # Every disturbed forest pixel must be classified
        if not np.array_equal(
            classification > 0,
            forest & any_disturbance
        ):
            raise RuntimeError(
                f"Incomplete classification in block {block_number}"
            )

        # Save sequence raster
        seq_dst.write(
            classification,
            1,
            window=window
        )

        # Save first-observed-year rasters
        for d, dst in year_dsts.items():

            output = first_year[d].copy()

            output[output == MISSING_YEAR] = (
                OUTPUT_YEAR_NODATA
            )

            dst.write(
                output,
                1,
                window=window
            )

        # ---------------------------------------------------------------------
        # Westwide sequence counts
        # ---------------------------------------------------------------------

        forest_total += int(
            np.count_nonzero(forest)
        )

        codes, counts = np.unique(
            classification[forest],
            return_counts=True
        )

        for code, count in zip(codes, counts):

            west_counts[int(code)] += int(count)

        # ---------------------------------------------------------------------
        # Ecoregion counts
        # ---------------------------------------------------------------------

        valid = forest & valid_eco

        eco_ids, eco_n = np.unique(
            eco[valid],
            return_counts=True
        )

        for eco_id, n in zip(eco_ids, eco_n):

            eco_forest_counts[int(eco_id)] += int(n)

        # Count sequence classes within each ecoregion
        for code in range(26):

            code_mask = valid & (
                classification == code
            )

            if not code_mask.any():
                continue

            ids, counts = np.unique(
                eco[code_mask],
                return_counts=True
            )

            for eco_id, n in zip(ids, counts):

                eco_counts[
                    (int(eco_id), code)
                ] += int(n)

        # ---------------------------------------------------------------------
        # Consecutive transitions and their time lags
        # ---------------------------------------------------------------------

        for code, groups in CODE_TO_GROUPS.items():

            if len(groups) < 2:
                continue

            pixel_mask = (
                classification == code
            )

            if not pixel_mask.any():
                continue

            n_pixels = int(
                np.count_nonzero(pixel_mask)
            )

            for transition_number, (
                from_group,
                to_group
            ) in enumerate(
                zip(groups[:-1], groups[1:]),
                start=1
            ):

                from_stage = stage_label(from_group)
                to_stage = stage_label(to_group)

                transition_key = (
                    code,
                    transition_number,
                    from_stage,
                    to_stage
                )

                transition_counts[
                    transition_key
                ] += n_pixels

                # All disturbances within a stage share a year
                from_year = first_year[from_group[0]][pixel_mask]
                to_year = first_year[to_group[0]][pixel_mask]

                lags = (
                    to_year.astype(np.int32)
                    - from_year.astype(np.int32)
                )

                if not np.all(lags > 0):
                    raise RuntimeError(
                        "Invalid nonpositive transition lag"
                    )

                lag_values, lag_counts = np.unique(
                    lags,
                    return_counts=True
                )

                for lag, n in zip(
                    lag_values,
                    lag_counts
                ):

                    transition_lag_counts[
                        (
                            code,
                            transition_number,
                            from_stage,
                            to_stage,
                            int(lag)
                        )
                    ] += int(n)


print("\nRaster processing complete.")


# =============================================================================
# 6. WESTWIDE SEQUENCE SUMMARY
# =============================================================================

disturbed_total = sum(
    west_counts[code]
    for code in range(1, 26)
)

west_rows = []

for code in range(26):

    n = west_counts[code]

    west_rows.append([
        code,
        CODE_TO_LABEL[code],
        CODE_TO_N_TYPES[code],
        n,
        n * PIXEL_AREA_KM2,
        100 * n / forest_total if forest_total else 0,
        (
            100 * n / disturbed_total
            if code > 0 and disturbed_total
            else 0
        )
    ])

save_csv(
    "extreme_sequences_westwide.csv",
    [
        "sequence_code",
        "sequence",
        "n_disturbance_types",
        "pixel_count",
        "area_km2",
        "percent_forest",
        "percent_disturbed_forest"
    ],
    west_rows
)


# =============================================================================
# 7. ECOREGION SEQUENCE SUMMARY
# =============================================================================

eco_rows = []

for eco_id in sorted(eco_forest_counts):

    forest_n = eco_forest_counts[eco_id]

    disturbed_n = sum(
        eco_counts[(eco_id, code)]
        for code in range(1, 26)
    )

    for code in range(26):

        n = eco_counts[(eco_id, code)]

        eco_rows.append([
            eco_id,
            code,
            CODE_TO_LABEL[code],
            CODE_TO_N_TYPES[code],
            n,
            n * PIXEL_AREA_KM2,
            100 * n / forest_n if forest_n else 0,
            (
                100 * n / disturbed_n
                if code > 0 and disturbed_n
                else 0
            )
        ])

save_csv(
    "extreme_sequences_ecoregions.csv",
    [
        "ecoregion_id",
        "sequence_code",
        "sequence",
        "n_disturbance_types",
        "pixel_count",
        "area_km2",
        "percent_ecoregion_forest",
        "percent_ecoregion_disturbed_forest"
    ],
    eco_rows
)


# =============================================================================
# 8. NUMBER OF DISTINCT EXTREME DISTURBANCE TYPES
# =============================================================================

disturbance_count_rows = []

for n_types in range(4):

    n = sum(
        west_counts[code]
        for code in range(26)
        if CODE_TO_N_TYPES[code] == n_types
    )

    disturbance_count_rows.append([
        n_types,
        n,
        n * PIXEL_AREA_KM2,
        100 * n / forest_total if forest_total else 0,
        (
            100 * n / disturbed_total
            if n_types > 0 and disturbed_total
            else 0
        )
    ])

save_csv(
    "extreme_disturbance_type_counts.csv",
    [
        "n_disturbance_types",
        "pixel_count",
        "area_km2",
        "percent_forest",
        "percent_disturbed_forest"
    ],
    disturbance_count_rows
)


# =============================================================================
# 9. ALLUVIAL-READY DATA
# =============================================================================

alluvial_rows = []

for code, groups in CODE_TO_GROUPS.items():

    n = west_counts[code]

    # Only disturbed sequences are included.
    # END means that no additional distinct disturbance
    # type was first observed during 2000–2020.

    stages = [
        stage_label(group)
        for group in groups
    ]

    stages += ["END"] * (3 - len(stages))

    alluvial_rows.append([
        code,
        CODE_TO_LABEL[code],
        CODE_TO_N_TYPES[code],
        stages[0],
        stages[1],
        stages[2],
        n,
        n * PIXEL_AREA_KM2,
        (
            100 * n / disturbed_total
            if disturbed_total
            else 0
        )
    ])

save_csv(
    "extreme_sequences_alluvial.csv",
    [
        "sequence_code",
        "sequence",
        "n_disturbance_types",
        "stage_1",
        "stage_2",
        "stage_3",
        "pixel_count",
        "area_km2",
        "percent_disturbed_forest"
    ],
    alluvial_rows
)


# =============================================================================
# 10. TRANSITION SUMMARY
# =============================================================================

# Detailed transition records by sequence.
transition_detail_rows = []

for key in sorted(transition_counts):

    code, number, from_stage, to_stage = key

    n = transition_counts[key]

    transition_detail_rows.append([
        code,
        CODE_TO_LABEL[code],
        number,
        from_stage,
        to_stage,
        n,
        n * PIXEL_AREA_KM2
    ])

save_csv(
    "extreme_transitions_by_sequence.csv",
    [
        "sequence_code",
        "sequence",
        "transition_number",
        "from_stage",
        "to_stage",
        "pixel_count",
        "area_km2"
    ],
    transition_detail_rows
)


# Aggregate transitions across sequence classes.
# A pixel can contribute more than one transition.

transition_totals = Counter()

for key, n in transition_counts.items():

    code, number, from_stage, to_stage = key

    transition_totals[
        (from_stage, to_stage)
    ] += n

transition_rows = []

for (from_stage, to_stage), n in sorted(
    transition_totals.items()
):

    transition_rows.append([
        from_stage,
        to_stage,
        n,
        n * PIXEL_AREA_KM2,
        (
            100 * n / disturbed_total
            if disturbed_total
            else 0
        )
    ])

save_csv(
    "extreme_transitions_westwide.csv",
    [
        "from_stage",
        "to_stage",
        "transition_pixel_count",
        "transition_area_km2",
        "percent_disturbed_forest_pixels"
    ],
    transition_rows
)


# =============================================================================
# 11. TRANSITION LAG DISTRIBUTIONS
# =============================================================================

# Detailed lag records
lag_detail_rows = []

# Aggregated lag counts by transition and number of years
lag_totals = Counter()

for key, n in sorted(
    transition_lag_counts.items()
):

    (
        code,
        number,
        from_stage,
        to_stage,
        lag
    ) = key

    lag_detail_rows.append([
        code,
        CODE_TO_LABEL[code],
        number,
        from_stage,
        to_stage,
        lag,
        n,
        n * PIXEL_AREA_KM2
    ])

    lag_totals[
        (from_stage, to_stage, lag)
    ] += n

save_csv(
    "extreme_transition_lags_by_sequence.csv",
    [
        "sequence_code",
        "sequence",
        "transition_number",
        "from_stage",
        "to_stage",
        "lag_years",
        "pixel_count",
        "area_km2"
    ],
    lag_detail_rows
)


lag_rows = []

for (from_stage, to_stage, lag), n in sorted(
    lag_totals.items()
):

    total_for_transition = transition_totals[
        (from_stage, to_stage)
    ]

    lag_rows.append([
        from_stage,
        to_stage,
        lag,
        n,
        n * PIXEL_AREA_KM2,
        (
            100 * n / total_for_transition
            if total_for_transition
            else 0
        )
    ])

save_csv(
    "extreme_transition_lags_westwide.csv",
    [
        "from_stage",
        "to_stage",
        "lag_years",
        "pixel_count",
        "area_km2",
        "percent_of_transition"
    ],
    lag_rows
)


# =============================================================================
# 12. FINAL VALIDATION
# =============================================================================

assert sum(west_counts.values()) == forest_total

assert (
    sum(
        row[1]
        for row in disturbance_count_rows
    )
    == forest_total
)

assert (
    sum(
        transition_counts.values()
    )
    == sum(
        transition_lag_counts.values()
    )
)

# Each sequence contributes one transition per pair
# of consecutive stages.
expected_transition_count = sum(
    west_counts[code] * (len(groups) - 1)
    for code, groups in CODE_TO_GROUPS.items()
)

assert (
    sum(transition_counts.values())
    == expected_transition_count
)

print("\n" + "=" * 65)
print("EXTREME DISTURBANCE SEQUENCE ANALYSIS COMPLETE")
print("=" * 65)

print(
    f"Total forest area: "
    f"{forest_total * PIXEL_AREA_KM2:,.1f} km²"
)

print(
    f"Forest area with ≥1 extreme disturbance: "
    f"{disturbed_total * PIXEL_AREA_KM2:,.1f} km²"
)

print(
    f"Percent of forest disturbed: "
    f"{100 * disturbed_total / forest_total:.2f}%"
)

print("\nNumber of extreme disturbance types:")

for n_types in range(4):

    n = disturbance_count_rows[n_types][1]

    print(
        f"  {n_types} types: "
        f"{n * PIXEL_AREA_KM2:,.1f} km²"
    )

print(
    "\nTotal consecutive first-occurrence transitions: "
    f"{sum(transition_counts.values()):,}"
)

print("\nAll outputs saved to:")
print(OUT_DIR)

Output directory: /home/jovyan/data-store/forest-disturbance-stack-v3/data/derived/disturbance_sequence/extreme_sequences
Saved: sequence_codebook.csv
Processed 100 blocks
Processed 200 blocks
Processed 300 blocks
Processed 400 blocks
Processed 500 blocks
Processed 600 blocks
Processed 700 blocks
Processed 800 blocks
Processed 900 blocks
Processed 1,000 blocks
Processed 1,100 blocks
Processed 1,200 blocks
Processed 1,300 blocks
Processed 1,400 blocks
Processed 1,500 blocks
Processed 1,600 blocks
Processed 1,700 blocks
Processed 1,800 blocks
Processed 1,900 blocks
Processed 2,000 blocks
Processed 2,100 blocks
Processed 2,200 blocks
Processed 2,300 blocks
Processed 2,400 blocks
Processed 2,500 blocks
Processed 2,600 blocks
Processed 2,700 blocks
Processed 2,800 blocks
Processed 2,900 blocks
Processed 3,000 blocks
Processed 3,100 blocks
Processed 3,200 blocks
Processed 3,300 blocks
Processed 3,400 blocks
Processed 3,500 blocks
Processed 3,600 blocks
Processed 3,700 blocks
Processed 3,800 